In [5]:
import pandas as pd
import altair as alt

if not hasattr(alt.utils, 'use_signature_func'):
    def _use_signature_func(_signature_func):
        def _decorator(func):
            return func
        return _decorator
    alt.utils.use_signature_func = _use_signature_func

from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme(theme_name="article")

In [47]:
df = pd.read_excel(
    'football_shirts/Data/Football shirt prices - original.xlsx',
    sheet_name='Table for chart'
).rename(columns={
    'League average shirt price': 'shirt_price',
    'Inflation-adjusted 2014/15 price': 'inflation_price',
})[['Season', 'shirt_price', 'inflation_price']]

L_SHIRT = 'League average shirt price'
L_INFL  = 'inflation-adjusted \'14/15 price'

# long format for the two lines
long = (
    df.melt(id_vars='Season',
            value_vars=['shirt_price', 'inflation_price'],
            var_name='series', value_name='price')
      .assign(label=lambda d: d['series'].map(
          {'shirt_price': L_SHIRT, 'inflation_price': L_INFL}))
)

In [13]:
styles.eco_colours

{'pink': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4',
 'green': '#00a767',
 'mid-blue': '#0063af',
 'purple': '#5c267b',
 'dot': '#f4134d',
 'grey': '#676a86'}

In [36]:
GREEN, GREY, PINK = '#00a767', '#676a86', '#e6224b'
WIDTH = 640

xenc = alt.X('Season:N',
             axis=alt.Axis(title=None, labelAngle=0, labelFontSize=14,
                           labelExpr='"\'" + slice(datum.label, 2)'),
             scale=alt.Scale(padding=0))
yscale = alt.Scale(domain=[40, 80])
yaxis  = alt.Axis(values=[40, 45, 50, 55, 60, 65, 70, 75, 80],
                  labelExpr='"£" + datum.value', labelFontSize=14, title=None)

# pink band between inflation-adjusted price (lower) and actual price (upper)
area = alt.Chart(df).mark_area(color=PINK, opacity=.1).encode(
    x=xenc,
    y=alt.Y('inflation_price:Q', scale=yscale, axis=yaxis),
    y2='shirt_price:Q',
)

# the two lines: solid green + dashed grey
lines = alt.Chart(long).mark_line(strokeWidth=2.5).encode(
    x=xenc,
    y=alt.Y('price:Q', scale=yscale, axis=yaxis),
    color=alt.Color('label:N',
        scale=alt.Scale(domain=[L_SHIRT, L_INFL], range=[GREEN, GREY]),
        legend=None),
    strokeDash=alt.StrokeDash('label:N',
            scale=alt.Scale(domain=[L_SHIRT, L_INFL], range=[[1, 0], [6, 4]]),
            legend=None),

)

main = alt.layer(area, lines).properties(width=WIDTH, height=360)

In [48]:
GREEN, GREY, PINK = '#00a767', '#676a86', '#e6224b'
WIDTH = 640

xenc = alt.X('Season:N',
             axis=alt.Axis(title=None, labelAngle=0, labelFontSize=14,
                           labelExpr='"\'" + slice(datum.label, 2)'),
             scale=alt.Scale(padding=0))
yscale = alt.Scale(domain=[40, 80])
yaxis  = alt.Axis(values=[40, 45, 50, 55, 60, 65, 70, 75, 80],
                  labelExpr='"£" + datum.value', labelFontSize=14, title=None)

colscale = alt.Scale(domain=[L_SHIRT, L_INFL], range=[GREEN, GREY])

# pink band between inflation-adjusted price (lower) and actual price (upper)
area = alt.Chart(df).mark_area(color=PINK, opacity=.1).encode(
    x=xenc,
    y=alt.Y('inflation_price:Q', scale=yscale, axis=yaxis),
    y2='shirt_price:Q',
)

# the two lines: solid green + dashed grey
lines = alt.Chart(long).mark_line(strokeWidth=2.5).encode(
    x=xenc,
    y=alt.Y('price:Q', scale=yscale, axis=yaxis),
    color=alt.Color('label:N', scale=colscale, legend=None),
    strokeDash=alt.StrokeDash('label:N',
        scale=alt.Scale(domain=[L_SHIRT, L_INFL], range=[[1, 0], [6, 4]]),
        legend=None),
)

# end-of-line labels, wrapped onto two lines, coloured to match each line
end_pts = long[long['Season'] == long['Season'].iloc[-1]]

label_lines = {                                    # each label split across two lines
    L_SHIRT: ['League average', 'shirt price'],
    L_INFL:  ['Inflation-adjusted', '2014/15 price'],
}
label_dy = {L_SHIRT: -14, L_INFL: 6}               # nudge each label up/down to avoid overlap

end_labels = alt.layer(*[
    alt.Chart(end_pts[end_pts.label == lbl]).mark_text(
        align='left', dx=8, dy=label_dy[lbl],
        fontWeight='bold', fontSize=13, lineHeight=15
    ).encode(
        x=xenc,
        y=alt.Y('price:Q', scale=yscale),
        text=alt.value(label_lines[lbl]),          # list -> multi-line text
        color=alt.Color('label:N', scale=colscale, legend=None),
    )
    for lbl in [L_SHIRT, L_INFL]
])

main = alt.layer(area, lines, end_labels).properties(
    width=WIDTH, height=360
).configure_view(stroke=None)

main

alt.LayerChart(...)

In [50]:
main.save('shirt_prices_vs_inflation.png', scale_factor=2)
main.save('shirt_prices_vs_inflation.json')